# autoresearch · 可复现分析 notebook

> 让 AI 代理过夜跑 LLM 训练实验 · Karpathy 的 agent-driven research 样板 repo

**完整报告**：[`autoresearch.html`](autoresearch.html)（35K / ~8.2K 字，含横纵分析 + 竞品对比 + 交汇总结）

本 notebook 让你用 git 命令复现报告里的关键证据。先 clone：
```bash
git clone https://github.com/karpathy/autoresearch ~/auto-research/autoresearch
```

## 执行摘要
- 35 commits / 3 周 / 10 贡献者
- 核心哲学：**人改 `program.md` / agent 改 `train.py`** 的分权宪法
- 最重要事件：initial commit 当天**自己删除了 v0 的 `spawn.sh`**（多-agent 编排），从多代理撤回单代理
- 真正难度不是算法，是 **agent runtime robustness**（NaN/traceback/无数据/FA3 fallback）
- 成果被 nanochat leaderboard 第 5、6 行直接收录（round 1 / round 2）

In [ ]:
import subprocess
from pathlib import Path
from collections import Counter

REPO = Path.home() / 'auto-research' / 'autoresearch'
def git(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True)

print('repo:', REPO)
print('commits:', git('rev-list', '--count', 'HEAD').strip())
print('range:', git('log', '--pretty=format:%ad', '--date=short').strip().splitlines()[-1],
      '→', git('log', '--pretty=format:%ad', '--date=short').strip().splitlines()[0])

## 1. 文件热点 — 为什么 README / program.md 是重点

如果 autoresearch 是一个正常的 ML repo，你期望 `train.py` 被改最多。但它不是——"上下文工程"才是第一等公民。

In [ ]:
names = [n for n in git('log', '--name-only', '--pretty=format:').splitlines() if n]
for f, c in Counter(names).most_common():
    bar = '█' * c
    print(f'  {c:>3}  {f:<20}  {bar}')

**读这张表**：README 改 15 次、`program.md` 改 9 次、`train.py` 只改 5 次。**上下文（给人 + 给 agent 的说明书）比代码更常被更新**。

## 2. 首日的自我否定 — spawn.sh 之死

Initial commit 包含 248 行 `spawn.sh`（多 agent 编排），同一天内被删。这是整个 repo 最重要的设计决策。

In [ ]:
# initial commit 里包含 spawn.sh 吗？
first = git('rev-list', '--max-parents=0', 'HEAD').strip()
initial_files = git('show', '--name-only', '--format=', first).strip().splitlines()
print('initial commit:', first[:7])
print('contained spawn.sh?', 'spawn.sh' in initial_files)
print('total files in initial:', len(initial_files))

# spawn.sh 的生命周期
life = git('log', '--diff-filter=AD', '--pretty=format:%h %ad %s', '--date=short', '--', 'spawn.sh')
print('\nspawn.sh lifecycle:')
print(life)

In [ ]:
# 当天所有 commits（chronological）
log = git('log', '--date=format:%Y-%m-%d %H:%M', '--pretty=format:%h | %ad | %s').splitlines()
day0 = [l for l in log if '2026-03-06' in l]
for l in reversed(day0):
    print(l)

**决策逻辑**（从 diff 推断）：多 agent 并发改 `train.py`、共享 `results.tsv`、分支协调——代价远超 1 GPU 并行的收益。`program.md` 去掉 spawn.sh 分支后，"一页剧本一个主语"才真正成立。

**学到的**：**承认错误的速度是一种竞争优势**。Karpathy 几小时内就回撤 v0 野心版，不给自己"再观察几天"的借口。

## 3. Agent runtime 鲁棒性才是真正的难点

一个典型假设：`autoresearch` 的难点是让 agent 变聪明。git history 否决了这个假设——6 条 commits 全在堵"代理呆坐的盲区"，而且都是社区贡献（2 人独立修同一处 NaN 问题！）。

In [ ]:
# 'agent 呆坐' 类 commits
patterns = {
    'NaN fast-fail':       ['NaN', 'fast-fail'],
    'traceback 强制读':     ['traceback', 'stack trace'],
    '无数据无限循环':       ['infinite loop', 'no training shards'],
    'FA3 跨硬件 fallback':  ['FA3', 'Hopper', 'fallback'],
    'download 并发':        ['download-workers', 'download_workers'],
}
all_log = git('log', '--pretty=format:%h %ad %s', '--date=short').splitlines()
for label, keywords in patterns.items():
    hits = [l for l in all_log if any(k.lower() in l.lower() for k in keywords)]
    if hits:
        print(f'\n### {label}')
        for l in hits:
            print(' ', l)

## 4. `program.md` 演化 — 从多 agent 到单 agent

In [ ]:
# program.md 总改动次数
commits_on_program = git('log', '--pretty=format:%h %ad %s', '--date=short', '--', 'program.md').splitlines()
print(f'program.md 总修改次数: {len(commits_on_program)}\n')
for l in commits_on_program:
    print(' ', l)

In [ ]:
# 最关键的一条：69eb7f9 cleanup more references to spawn.sh
# 它把 program.md 从"单/多 agent 双路径"改成"只有单 agent"
print(git('show', '--stat', '69eb7f9').split('\n\n', 1)[0])
print('\n--- diff excerpt ---')
diff = git('show', '69eb7f9', '--', 'program.md')
print('\n'.join(diff.splitlines()[:30]))

## 5. 作者 + 社区贡献结构

In [ ]:
authors = git('log', '--pretty=format:%an').splitlines()
for a, n in Counter(authors).most_common():
    print(f'  {n:>3}  {a}')

**观察**：28/35 条来自 Karpathy，7 条来自 9 个社区贡献者——但这 7 条全是**核心鲁棒性修复**（FA3 fallback / NaN / download workers / analysis fix）。这是一个健康的 "小 repo + 快速反馈环" 模型。

## 6. 与主线项目的连接 — autoresearch → nanochat

如果你 clone 了 nanochat，可以验证 autoresearch round 1/2 的 commit 真的进了 nanochat leaderboard。

In [ ]:
nanochat_path = Path.home() / 'auto-research' / 'nanochat'
if nanochat_path.is_dir():
    out = subprocess.check_output(
        ['git', '-C', str(nanochat_path), 'log', '-i', '--grep=autoresearch',
         '--pretty=format:%h %ad %s', '--date=short'], text=True)
    print(out)
else:
    print('clone nanochat 先跑这个 cell')
    print('git clone https://github.com/karpathy/nanochat ~/auto-research/nanochat')

## 7. Takeaways

1. **分权宪法**是核心哲学：人改 md / agent 改 py。评估代码 read-only 是防 reward hacking 的物理护栏。
2. **首日回撤 spawn.sh** 比任何后续改动重要——这是一次现场的"自我否定"，定义了整个项目的约束空间。
3. **Agent runtime robustness** 是自主实验的真正 bottleneck，不是算法能力。建 program.md 时要假设 agent 最差 case。
4. **小 repo + 大杠杆**：被 nanochat 主线直接采用两次，是 2026 年 autonomous AI research 最干净的 proof point。

完整分析见 [`autoresearch.html`](autoresearch.html)。